# k-Nearest-Neighbors von Grund auf

**k-NN-Algorithmus komplett selbst implementiert — Einfluss des Parameters k**

In diesem Notebook implementieren wir den k-Nearest-Neighbors-Klassifikator ohne externe ML-Bibliotheken.
Wir untersuchen den Einfluss des Parameters **k** auf die Decision Boundary und die Klassifikationsgenauigkeit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from knn_from_scratch import KNearestNeighbors
from plot_utils import plot_decision_boundary

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Daten generieren

Wir verwenden einen synthetischen 2D-Datensatz, um die Decision Boundary gut visualisieren zu können.

In [ ]:
# Synthetische Daten: zwei Halbmonde
X, y = make_moons(n_samples=300, noise=0.25, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Trainingsdaten: {X_train.shape[0]} Samples")
print(f"Testdaten:      {X_test.shape[0]} Samples")
print(f"Klassen:        {np.unique(y)}")

In [ ]:
# Daten visualisieren
fig, ax = plt.subplots()
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='k')
ax.set_xlabel('Merkmal 1')
ax.set_ylabel('Merkmal 2')
ax.set_title('Synthetischer Datensatz: Two Moons')
plt.colorbar(scatter, label='Klasse')
plt.show()

## 2. Der k-NN-Algorithmus

### Funktionsweise

k-Nearest-Neighbors ist ein **Lazy Learner**: Es gibt keine explizite Trainingsphase. Stattdessen werden alle Trainingsdaten gespeichert und bei der Vorhersage die **k nächsten Nachbarn** gesucht.

**Ablauf:**
1. Für einen neuen Punkt $x$: Berechne die Distanz zu **allen** Trainingspunkten
2. Wähle die $k$ Punkte mit der kleinsten Distanz
3. **Mehrheitsentscheidung**: Die am häufigsten vorkommende Klasse unter den $k$ Nachbarn ist die Vorhersage

**Distanzmetriken:**
- **Euklidisch:** $d = \sqrt{\sum (x_i - y_i)^2}$ — „Luftlinie"
- **Manhattan:** $d = \sum |x_i - y_i|$ — „Straßenblocksumme"

## 3. k-NN mit k=1 (nächster Nachbar)

In [ ]:
# k-NN mit k=1 (euklidisch)
knn1 = KNearestNeighbors(k=1, metric='euclidean')
knn1.fit(X_train, y_train)

y_pred_1 = knn1.predict(X_test)
acc_1 = accuracy_score(y_test, y_pred_1)
print(f"Genauigkeit (k=1): {acc_1:.4f}")
print()
print(classification_report(y_test, y_pred_1, target_names=['Klasse 0', 'Klasse 1']))

In [ ]:
# Decision Boundary für k=1
fig, ax = plt.subplots()
plot_decision_boundary(knn1, X_train, y_train,
                       f'k-NN mit k=1 (Genauigkeit: {acc_1:.3f})', ax=ax)
plt.show()

**Beobachtung:** Mit k=1 ist die Decision Boundary sehr „zackig" und passt sich stark an einzelne Trainingspunkte an — hohes **Overfitting**-Risiko.

## 4. k-NN mit k=5

In [ ]:
# k-NN mit k=5
knn5 = KNearestNeighbors(k=5, metric='euclidean')
knn5.fit(X_train, y_train)

y_pred_5 = knn5.predict(X_test)
acc_5 = accuracy_score(y_test, y_pred_5)
print(f"Genauigkeit (k=5): {acc_5:.4f}")
print()
print(classification_report(y_test, y_pred_5, target_names=['Klasse 0', 'Klasse 1']))

In [ ]:
# Decision Boundary für k=5
fig, ax = plt.subplots()
plot_decision_boundary(knn5, X_train, y_train,
                       f'k-NN mit k=5 (Genauigkeit: {acc_5:.3f})', ax=ax)
plt.show()

## 5. k-NN mit k=15

In [ ]:
# k-NN mit k=15
knn15 = KNearestNeighbors(k=15, metric='euclidean')
knn15.fit(X_train, y_train)

y_pred_15 = knn15.predict(X_test)
acc_15 = accuracy_score(y_test, y_pred_15)
print(f"Genauigkeit (k=15): {acc_15:.4f}")
print()
print(classification_report(y_test, y_pred_15, target_names=['Klasse 0', 'Klasse 1']))

In [ ]:
# Decision Boundary für k=15
fig, ax = plt.subplots()
plot_decision_boundary(knn15, X_train, y_train,
                       f'k-NN mit k=15 (Genauigkeit: {acc_15:.3f})', ax=ax)
plt.show()

## 6. Einfluss von k: Systematischer Vergleich

Wie verändert sich die Genauigkeit mit steigendem k?

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]
train_scores = []
test_scores = []

for k in k_values:
    knn = KNearestNeighbors(k=k, metric='euclidean')
    knn.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, knn.predict(X_train)))
    test_scores.append(accuracy_score(y_test, knn.predict(X_test)))

fig, ax = plt.subplots()
ax.plot(k_values, train_scores, 'o-', label='Training', linewidth=2)
ax.plot(k_values, test_scores, 's-', label='Test', linewidth=2)
ax.set_xlabel('k (Anzahl Nachbarn)')
ax.set_ylabel('Genauigkeit')
ax.set_title('Einfluss von k auf die Klassifikationsgenauigkeit')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

**Interpretation:**
- **k=1**: Training = 100% (perfekt), aber Test fällt ab → Overfitting
- **k zu groß**: Training und Test fallen beide → Underfitting (zu starke Glättung)
- **Optimales k**: Bester Kompromiss zwischen Bias und Varianz

## 7. Decision Boundaries für verschiedene k-Werte

In [ ]:
k_show = [1, 3, 7, 15]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, k in enumerate(k_show):
    knn = KNearestNeighbors(k=k, metric='euclidean')
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    plot_decision_boundary(knn, X_train, y_train,
                          f'k={k} (Test-Acc: {acc:.3f})', ax=axes[i])

plt.suptitle('Decision Boundaries für verschiedene k-Werte', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 8. Euklidische vs. Manhattan-Distanz

In [ ]:
# Vergleich der Distanzmetriken
knn_euc = KNearestNeighbors(k=5, metric='euclidean')
knn_euc.fit(X_train, y_train)
acc_euc = accuracy_score(y_test, knn_euc.predict(X_test))

knn_man = KNearestNeighbors(k=5, metric='manhattan')
knn_man.fit(X_train, y_train)
acc_man = accuracy_score(y_test, knn_man.predict(X_test))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_decision_boundary(knn_euc, X_train, y_train,
                       f'Euklidisch (Genauigkeit: {acc_euc:.3f})', ax=axes[0])
plot_decision_boundary(knn_man, X_train, y_train,
                       f'Manhattan (Genauigkeit: {acc_man:.3f})', ax=axes[1])
plt.tight_layout()
plt.show()

## 9. Klassenwahrscheinlichkeiten (predict_proba)

Statt harter Klassenlabels können wir auch Wahrscheinlichkeiten ausgeben — nützlich für Konfidenzschätzungen.

In [ ]:
# Wahrscheinlichkeiten für die ersten 5 Testpunkte
proba = knn5.predict_proba(X_test[:5])

print("Klassenwahrscheinlichkeiten (erste 5 Testpunkte):")
print("=" * 50)
for i in range(5):
    print(f"Punkt {i}: Klasse 0 = {proba[i, 0]:.2f}, Klasse 1 = {proba[i, 1]:.2f}  →  "
          f"Vorhersage: Klasse {knn5.predict(X_test[i:i+1])[0]}")

## 10. Zusammenfassung

- **k-NN** ist ein einfacher, aber mächtiger Klassifikator — kein Training nötig, alle Daten werden gespeichert.
- **k** ist der zentrale Hyperparameter:
  - **k=1**: Overfitting, rauschempfindlich
  - **k groß**: Underfitting, zu starke Glättung
  - **Optimales k**: Balanciert Bias und Varianz (oft $k \approx \sqrt{n}$)
- **Distanzmetrik** beeinflusst die Form der Decision Boundary (euklidisch → rund, Manhattan → rautenförmig).
- **Nachteil**: Langsam bei großen Datensätzen — jede Vorhersage berechnet Distanzen zu allen Trainingspunkten ($O(n \cdot d)$).
- Die Implementierung verwendet `np.argpartition` für effiziente k-nächste-Nachbarn-Suche ohne vollständiges Sortieren.